In [3]:
# Cell 1, Impor Pustaka & Helper

# Impor Pustaka Eksternal
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options  
from webdriver_manager.chrome import ChromeDriverManager

# Impor Modul Lokal
# --- PENTING!!!: Impor fungsi dari file 'scraper_helper.py' ---
from scraper_helper import scrape_year_data

(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)


In [ ]:
# Cell 2: Inisiasi Driver

def setup_stealth_driver():
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-web-security")
    options.add_argument("--disable-features=VizDisplayCompositor")

    service = ChromeService(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)

    # Hapus window.navigator.webdriver
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    return driver


print("Memulai Chrome Driver dengan stealth mode...")

driver = setup_stealth_driver()
driver.maximize_window()

driver.get("https://x.com/login")
print("Driver dimulai. Browser telah terbuka. Silakan login secara manual.")

Memulai Chrome Driver dengan stealth mode...
Driver dimulai. Browser telah terbuka. Silakan login secara manual.


In [ ]:
# Cell 3: Pusat Kendali Scraping (Eksekusi per Tahun)

TAHUN_TARGET = 2015
MAX_SCROLLS = 1000

if 'driver' in locals() and driver.session_id:
    print(f"======= MEMULAI PROSES SCRAPING TAHUN {TAHUN_TARGET} =======")

    try:
        # --- HANYA MEMANGGIL FUNGSI ---
        # Semua logika rumit ada di scraper_helper.py
        scrape_year_data(TAHUN_TARGET, driver, max_scrolls=MAX_SCROLLS)

        print(f"\n======= SELESAI SCRAPING TAHUN {TAHUN_TARGET} =======\n")
    except Exception as e:
        print(f"\n--- Terjadi Error saat scraping tahun {TAHUN_TARGET}: {e} ---")
        print("Proses dihentikan. Driver tetap terbuka untuk inspeksi.")

else:
    print("ERROR: Driver tidak ditemukan. Harap jalankan 'Cell 2' terlebih dahulu dan selesaikan login manual.")

======= MEMULAI PROSES SCRAPING TAHUN 2015 =======

--- Memulai Scraping Tahun 2015. Target URL: https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2015-01-01%20until%3A2015-12-31&src=typed_query&f=live ---
Tahun 2015 | Scroll ke-1 | Data unik terkumpul: 0
Tahun 2015 | Scroll ke-2 | Data unik terkumpul: 12


In [4]:
# Cell 4: Tutup Driver 
try:
    driver.quit()
    print("Driver telah berhasil ditutup.")
except Exception as e:
    print(f"Gagal menutup driver (mungkin sudah ditutup): {e}")

Driver telah berhasil ditutup.


In [ ]:
# Cell 5: Pengecekan Total Data
import csv
import os

tahun_target = ["2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024", "2025"]
total_data_global = 0

print("--- Hasil Total Data Scraping (per File) ---")

for year in tahun_target:
    filename = f"aspirasi_dpr_{year}.csv"

    # Cek apakah file-nya ada
    if os.path.exists(filename):
        try:
            with open(filename, 'r', newline='', encoding='utf-8') as f:
                # Membaca semua baris
                reader = csv.reader(f)
                # Menghitung jumlah baris (dikurangi 1 untuk header)
                data_count = len(list(reader)) - 1

                # Pastikan data_count tidak negatif jika file kosong
                if data_count < 0:
                    data_count = 0

                print(f"Dataset Tahun {year}: {data_count} baris")
                total_data_global += data_count
        except Exception as e:
            print(f"Gagal membaca file {filename}: {e}")
    else:
        # Beri tahu jika file-nya tidak ditemukan
        print(f"File {filename} tidak ditemukan (dilewati).")

print("\n")
print(f"TOTAL DATA KESELURUHAN: {total_data_global} baris")

--- Hasil Total Data Scraping (per File) ---
Dataset Tahun 2016: 6204 baris
Dataset Tahun 2017: 5290 baris
Dataset Tahun 2018: 2387 baris
Dataset Tahun 2019: 392 baris
Dataset Tahun 2020: 517 baris
Dataset Tahun 2021: 636 baris
Dataset Tahun 2022: 30 baris
Dataset Tahun 2023: 74 baris
Dataset Tahun 2024: 633 baris
Dataset Tahun 2025: 216 baris


TOTAL DATA KESELURUHAN: 16379 baris


In [1]:
import pandas as pd
import glob
import os

In [2]:
files = glob.glob('aspirasi_dpr_*.csv')

In [3]:
data_list = []

for f in files:
    # Membaca file
    df = pd.read_csv(f)
    
    # Mengambil tahun dari nama file (misal: dari 'aspirasi_dpr_2016.csv' ambil '2016')
    tahun = f.split('_')[-1].split('.')[0]
    df['tahun'] = tahun
    
    data_list.append(df)

In [4]:
master_df = pd.concat(data_list, ignore_index=True)

master_df.head(10)

,timestamp,username,text,url_pencarian,tahun
0,2016-12-30T06:37:56.000Z,@majudepan578,ADIL loh utk yg punya kebijakan publik & negar...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
1,2016-12-30T06:30:36.000Z,NaN,"Tertibkan Media Online, DPR: Pemerintah Jangan...",https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
2,2016-12-30T04:48:35.000Z,@jiggerbunny,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
3,2016-12-30T04:21:40.000Z,@marunduri_tri,"jangan ngambang, aturan logis apa undang-undan...",https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
4,2016-12-30T02:36:13.000Z,NaN,6. Kebebasan bersuara & berpendapat memang dij...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
5,2016-12-30T02:52:25.000Z,@DPR_RI,Anggota Komisi IX DPR Okky Asokawati: Evaluasi...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
6,2016-12-30T01:18:56.000Z,@Jasutra,Anggota DPR-RI Komisi II Dari Fraksi NasDem Ge...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
7,2016-12-30T00:50:51.000Z,@csps_indonesia,Wapres Minta Kebijakan Bebas Visa Dievaluasi h...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
8,2016-12-29T16:20:02.000Z,@tropongsenayan,"Asal Sesuai Undang-Undang, Anggota DPR Ini Set...",https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016
9,2016-12-29T11:52:05.000Z,@ambartjahyono60,Saat memberikan makanan bergizi bantuan Kemenk...,https://x.com/search?q=(%22DPR%20RI%22%20OR%20...,2016


In [14]:
from IPython.display import display

display(
    preview_per_tahun.style.set_properties(
        subset=['url_pencarian'],
        **{
            'white-space': 'normal',
            'word-wrap': 'break-word',
            'max-width': '600px'
        }
    )
)

,timestamp,username,text,url_pencarian,tahun
0,2016-12-30T06:37:56.000Z,@majudepan578,ADIL loh utk yg punya kebijakan publik & negara Ingat yg ini!!! @jokowi @ZUL_Hasan @DPR_RI @kemkominfo @rudiantara_id @sn_setyanovanto,https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2016-01-01%20until%3A2016-12-31&src=typed_query&f=live,2016
1,2016-12-30T06:30:36.000Z,nan,"Tertibkan Media Online, DPR: Pemerintah Jangan Sporadis, Apalagi Selektif Hanya kepada Media yang Berseberangan http://dlvr.it/N0GcNs",https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2016-01-01%20until%3A2016-12-31&src=typed_query&f=live,2016
2,2016-12-30T04:48:35.000Z,@jiggerbunny,"@Portal_Kemlu_RI @DPR_RI @jokowi harus dievaluasi lg kebijakan bebas visa, trutama utk negara tiongkok pak!! bahaya tersembunyi martabat RI",https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2016-01-01%20until%3A2016-12-31&src=typed_query&f=live,2016
3,2016-12-30T04:21:40.000Z,@marunduri_tri,"jangan ngambang, aturan logis apa undang-undang @DPR_RI",https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2016-01-01%20until%3A2016-12-31&src=typed_query&f=live,2016
4,2016-12-30T02:36:13.000Z,nan,"6. Kebebasan bersuara & berpendapat memang dijamin UU. Namun kebebasan tersebut tdk harus kebablasan, sehingga menabrak aturan-aturan logis.",https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2016-01-01%20until%3A2016-12-31&src=typed_query&f=live,2016
6204,2017-12-30T23:02:56.000Z,@Mohroml95025017,Bisa cuman berkoar koar fahri hamsah anda sebagai dewan perwakilan rakyat ( DPR ) RUU ZINA & LGBT tdk kunjung selesai sampai detik ini apa aja yg di kerjain anda.anda seorang muslim sekaligus wakil pemimpin sangat lah berdosa menurut agama,https://x.com/search?q=(%22DPR%20RI%22%20OR%20%22Dewan%20Perwakilan%20Rakyat%22%20OR%20%22anggota%20DPR%22%20OR%20%22wakil%20rakyat%22%20OR%20%22Komisi%20DPR%22%20OR%20%22Gedung%20DPR%22)%20(%22RUU%22%20OR%20%22undang-undang%22%20OR%20%22kebijakan%22%20OR%20%22pengesahan%22%20OR%20%22reses%22%20OR%20%22rapat%20paripurna%22%20OR%20%22aspirasi%20rakyat%22)%20since%3A2017-01-01%20until%3A2017-12-31&src=typed_query&f=live,2017
6205,2017-12-30T18:31:05.000Z,@edunewsid,"Refleksi Akhir Tahun, DPR Kurang Mendengar Aspirasi Rakyat: JAKARTA, EDUNEWS.ID – Ketua Majelis Permusyawaratan Rakyat (MPR) Zulkifli Hasan mengatakan publik merasa tidak percaya lagi terhadap Dewan Perwakilan Rakyat (DPR) dan partai politik selama… http://dlvr.it/

In [7]:
master_df.to_csv('master_dataset_aspirasi_dpr.csv', index=False)

print(f"Total data setelah digabung: {len(master_df)} baris.")

Total data setelah digabung: 16379 baris.
